# Unit 4 Assignment: Evaluated Agentic RAG System

**Name:** Bhavana Ramkumar
**SRN:** PES2UG23CS905

---

This notebook builds a **self-evaluating agentic RAG system** using:
- **LangChain + FAISS** for retrieval
- **CrewAI** for multi-agent orchestration
- **DeepEval** for automated answer quality scoring (Faithfulness + Answer Relevancy)
- **Groq** (LLaMA 3.3-70B) as the LLM backbone

The pipeline: RAG Agent → Evaluator Agent → (if FAIL) Revisor Agent → Final Answer


In [ ]:
!pip install -q langchain langchain-community langchain-groq faiss-cpu \
    sentence-transformers crewai crewai-tools deepeval groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 784.2/784.2 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 843.4/843.4 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

# ── Set your API keys ──────────────────────────────────────────────────────────
os.environ["GROQ_API_KEY"] = "API_KEY"
# ──────────────────────────────────────────────────────────────────────────────

print("Environment ready.")

Environment ready.


---
## Part 1 — Knowledge Base (Climate Change)

We embed a 600+ word knowledge base about climate change into a FAISS vector store using HuggingFace sentence-transformer embeddings.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# ── Knowledge Base Text ────────────────────────────────────────────────────────
KNOWLEDGE_BASE = """
Climate Change: Causes, Effects, and Scientific Consensus

Climate change refers to long-term shifts in global temperatures and weather patterns.
While some climate change occurs naturally, scientific evidence shows that since the
mid-20th century, human activities have been the dominant driver of observed warming.

## Greenhouse Gases
The primary cause of current climate change is the emission of greenhouse gases (GHGs),
particularly carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). CO2 levels
in the atmosphere have risen from approximately 280 parts per million (ppm) in the
pre-industrial era to over 420 ppm as of 2023 — the highest in at least 800,000 years.
Methane is over 80 times more potent than CO2 over a 20-year period as a greenhouse gas.

## Temperature Rise
The global average surface temperature has increased by approximately 1.1 degrees Celsius
since the late 19th century. The last decade (2011-2020) was the warmest on record.
The Paris Agreement, adopted in 2015 by 196 parties, aims to limit warming to 1.5°C
above pre-industrial levels to avoid the most severe climate impacts.

## Melting Ice and Sea Level Rise
The Arctic is warming at more than twice the global average rate — a phenomenon called
Arctic amplification. Since 1979, Arctic sea ice extent has declined by about 13% per
decade. Greenland and Antarctica are losing ice mass at accelerating rates. Global mean
sea level has risen by about 20 cm since 1900, with the rate of rise accelerating in
recent decades to about 3.7 mm per year.

## Ocean Changes
The oceans absorb about 90% of the excess heat trapped by greenhouse gases and roughly
25-30% of all CO2 emissions. Ocean acidification — caused by CO2 dissolving in seawater
— has lowered ocean pH by 0.1 units since industrialization, representing a 26% increase
in acidity. Coral reefs are highly sensitive to these changes; a 2°C warming scenario
is projected to destroy 99% of coral reefs worldwide.

## Extreme Weather Events
Climate change amplifies extreme weather events. Heatwaves have become more frequent
and intense. Hurricanes are becoming stronger and wetter due to warmer ocean surfaces.
Heavy precipitation events are increasing in many regions while droughts worsen in others.
Wildfires are burning larger areas globally, with fire seasons growing longer by an
average of 18.7% since 1979.

## Biodiversity and Ecosystems
Climate change is one of the leading drivers of biodiversity loss. Species are shifting
their ranges poleward and to higher elevations at average rates of 17 km per decade.
Phenological mismatches — where the timing of species interactions (e.g., flowering
and pollinator emergence) no longer aligns — are disrupting ecosystems. The IPCC warns
that 20-30% of species face increased extinction risk at 1.5-2°C of warming.

## Human Health Impacts
Rising temperatures expand the geographic range of disease vectors like mosquitoes,
increasing exposure to diseases such as malaria and dengue fever. Heat stress causes
thousands of deaths annually and reduces outdoor labour capacity. Food insecurity is
projected to worsen as crop yields decline under higher temperatures; wheat yields
decline by 6% for every 1°C of warming.

## Scientific Consensus
Over 97% of actively publishing climate scientists agree that current climate warming
trends are extremely likely due to human activities. This consensus is reflected in
reports by the Intergovernmental Panel on Climate Change (IPCC), NASA, NOAA, and
virtually every major scientific organization worldwide. The IPCC Sixth Assessment
Report (AR6), published in 2021-2022, describes the evidence as 'unequivocal'.

## Mitigation and Adaptation
Mitigation involves reducing GHG emissions through renewable energy, energy efficiency,
electrification of transport, and carbon capture. Adaptation involves adjusting to
current and projected climate impacts — for example, building sea walls, developing
drought-resistant crops, and redesigning urban infrastructure. Achieving net-zero
CO2 emissions by 2050 is considered the central pathway to limiting warming to 1.5°C.
"""

# ── Chunk the text ─────────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)
chunks = splitter.create_documents([KNOWLEDGE_BASE])
print(f"Created {len(chunks)} chunks from knowledge base")
for i, c in enumerate(chunks[:3]):
    print(f"\n--- Chunk {i+1} ---\n{c.page_content[:150]}...")

Created 20 chunks from knowledge base

--- Chunk 1 ---
Climate Change: Causes, Effects, and Scientific Consensus...

--- Chunk 2 ---
Climate change refers to long-term shifts in global temperatures and weather patterns.
While some climate change occurs naturally, scientific evidence...

--- Chunk 3 ---
## Greenhouse Gases
The primary cause of current climate change is the emission of greenhouse gases (GHGs),
particularly carbon dioxide (CO2), methane...


In [ ]:
# ── Build FAISS vector store ───────────────────────────────────────────────────
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

vector_store = FAISS.from_documents(chunks, embeddings)
retriever    = vector_store.as_retriever(search_kwargs={"k": 4})

print(f"FAISS index built — {vector_store.index.ntotal} vectors stored")

/tmp/ipykernel_2071/886646583.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

FAISS index built — 20 vectors stored


## Part 2: RAG Agent

The RAG agent uses a `@tool`-decorated function to query the FAISS vector store and
generates an answer grounded in the retrieved context. Its output always contains
**both the answer and the raw retrieved context** so the evaluator can run faithfulness checks.


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from langchain_groq import ChatGroq

# ── LLM (for tool) ───────────────────────────────────────────────────────────
# Instantiate llm for use within the tool (this one is Langchain's ChatGroq)
tool_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

# ── LLM (for agent) ──────────────────────────────────────────────────────────
# Configure CrewAI's LLM for the agent using a dictionary for an OpenAI-compatible model
# This uses the 'openai' llm_type and provides parameters compatible with Groq's OpenAI API.
agent_llm = {
    "llm_type": "openai",
    "model": "llama-3.3-70b-versatile", # CrewAI expects 'model' for its internal OpenAI config
    "api_key": os.environ["GROQ_API_KEY"],
    "base_url": "https://api.groq.com/openai/v1", # Groq's OpenAI-compatible base URL
    "temperature": 0.1
}

# ── RAG Tool ──────────────────────────────────────────────────────────────────
@tool("climate_rag_tool")
def climate_rag_tool(question: str) -> str:
    """
    Retrieves relevant context from the climate change knowledge base
    and generates a grounded answer. Returns both the answer and the
    raw retrieved context in a structured format.
    """
    # Retrieve
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])

    # Generate answer using LLM
    prompt = f"""You are a climate science expert. Answer the question below
using ONLY the provided context. If the context does not contain enough
information, say so explicitly.

Context:
{context}

Question: {question}

Answer:"""

    response = tool_llm.invoke(prompt) # The tool uses the langchain_groq instance
    answer   = response.content.strip()

    # Return structured output
    return f"""ANSWER:
{answer}

RETRIEVED_CONTEXT:
{context}"""


# ── RAG Agent ─────────────────────────────────────────────────────────────────
rag_agent = Agent(
    role="Climate Science RAG Specialist",
    goal="Retrieve relevant information from the climate change knowledge base "
         "and generate accurate, grounded answers to user questions.",
    backstory="You are an expert at information retrieval. You always use the "
              "climate_rag_tool to fetch context before answering, and you "
              "NEVER make up information not found in the retrieved documents.",
    tools=[climate_rag_tool],
    llm=agent_llm, # Use the CrewAI agent LLM configured as an OpenAI-compatible model
    verbose=True
)

print("RAG agent defined.")

RAG agent defined.


In [ ]:
def make_rag_task(question: str) -> Task:
    return Task(
        description=f"""Use the climate_rag_tool to answer the following question:

Question: {question}

Your final output MUST be formatted exactly like this:

ANSWER:
<your answer here>

RETRIEVED_CONTEXT:
<the context chunks retrieved from the vector store>""",
        expected_output="A structured response with ANSWER and RETRIEVED_CONTEXT sections.",
        agent=rag_agent
    )


# ── Quick test on 3 questions ─────────────────────────────────────────────────
test_questions = [
    "What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?",
    "How does climate change affect coral reefs?",
    "What is the Paris Agreement and what temperature target does it set?"
]

rag_test_outputs = {}

for q in test_questions:
    print(f"\n{'='*70}\nQ: {q}\n{'='*70}")
    task   = make_rag_task(q)
    crew   = Crew(agents=[rag_agent], tasks=[task], process=Process.sequential, verbose=False)
    result = crew.kickoff()
    rag_test_outputs[q] = str(result)
    print(result)


Q: What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Task: Use the climate_rag_tool to answer the following question:                                               │
│                                                                                                                 │
│  Question: What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?     │
│                                                                                                                 │
│  Your final output MUST be formatted exactly like this:                                                         │
│                                                                                                                 │
│  ANSWER:                                                                                                        │
│  <your answer here>                                                                                             │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  <the context chunks retrieved from the vector store>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool climate_rag_tool executed with result: ANSWER:
The current CO2 level in the atmosphere is over 420 ppm as of 2023. In comparison, pre-industrial CO2 levels were approximately 280 parts per million (ppm), indicating a significant increase.
...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER:                                                                                                        │
│  The current CO2 level in the atmosphere is over 420 ppm as of 2023. In comparison, pre-industrial CO2 levels   │
│  were approximately 280 parts per million (ppm), indicating a significant increase.                             │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  pre-industrial era to over 420 ppm as of 2023 — the highest in at least 800,000 years.                         │
│  Methane is over 80 times more potent than CO2 over a 20-year period as a greenhouse gas.                       │
│                                                                                                                 │
│  ## Greenhouse Gases                                                                                            │
│  The primary cause of current climate change is the emission of greenhouse gases (GHGs),                        │
│  particularly carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). CO2 levels                          │
│  in the atmosphere have risen from approximately 280 parts per million (ppm) in the                             │
│                                                                                                                 │
│  ## Ocean Changes                                                                                               │
│  The oceans absorb about 90% of the excess heat trapped by greenhouse gases and roughly                         │
│  25-30% of all CO2 emissions. Ocean acidification — caused by CO2 dissolving in seawater                        │
│  — has lowered ocean pH by 0.1 units since industrialization, representing a 26% increase                       │
│                                                                                                                 │
│  above pre-industrial levels to avoid the most severe climate impacts.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Task: Use the climate_rag_tool to answer the following question:                                               │
│                                                                                                                 │
│  Question: What is the Paris Agreement and what temperature target does it set?                                 │
│                                                                                                                 │
│  Your final output MUST be formatted exactly like this:                                                         │
│                                                                                                                 │
│  ANSWER:                                                                                                        │
│  <your answer here>                                                                                             │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  <the context chunks retrieved from the vector store>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool climate_rag_tool executed with result: ANSWER:
The Paris Agreement is an international agreement adopted in 2015 by 196 parties, and it aims to limit warming to 1.5°C.

RETRIEVED_CONTEXT:
## Temperature Rise
The global average surface temp...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER:                                                                                                        │
│  The Paris Agreement is an international agreement adopted in 2015 by 196 parties, and it aims to limit         │
│  warming to 1.5°C.                                                                                              │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  ## Temperature Rise                                                                                            │
│  The global average surface temperature has increased by approximately 1.1 degrees Celsius                      │
│  since the late 19th century. The last decade (2011-2020) was the warmest on record.                            │
│  The Paris Agreement, adopted in 2015 by 196 parties, aims to limit warming to 1.5°C                            │
│                                                                                                                 │
│  drought-resistant crops, and redesigning urban infrastructure. Achieving net-zero                              │
│  CO2 emissions by 2050 is considered the central pathway to limiting warming to 1.5°C.                          │
│                                                                                                                 │
│  Climate Change: Causes, Effects, and Scientific Consensus                                                      │
│                                                                                                                 │
│  ## Human Health Impacts                                                                                        │
│  Rising temperatures expand the geographic range of disease vectors like mosquitoes,                            │
│  increasing exposure to diseases such as malaria and dengue fever. Heat stress causes                           │
│  thousands of deaths annually and reduces outdoor labour capacity. Food insecurity is                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

ANSWER:
The Paris Agreement is an international agreement adopted in 2015 by 196 parties, and it aims to limit warming to 1.5°C.

RETRIEVED_CONTEXT:
## Temperature Rise
The global average surface temperature has increased by approximately 1.1 degrees Celsius 
since the late 19th century. The last decade (2011-2020) was the warmest on record.
The Paris Agreement, adopted in 2015 by 196 parties, aims to limit warming to 1.5°C

drought-resistant crops, and redesigning urban infrastructure. Achieving net-zero 
CO2 emissions by 2050 is considered the central pathway to limiting warming to 1.5°C.

Climate Change: Causes, Effects, and Scientific Consensus

## Human Health Impacts
Rising temperatures expand the geographic range of disease vectors like mosquitoes, 
increasing exposure to diseases such as malaria and dengue fever. Heat stress causes 
thousands of deaths annually and reduces outdoor labour capacity. Food insecurity is


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Part 3: Quality Evaluator Agent

The evaluator agent wraps DeepEval's `FaithfulnessMetric` and `AnswerRelevancyMetric`
inside a `@tool`. It reads the RAG output (answer + context), runs both metrics,
and returns a structured verdict with scores, pass/fail status, and specific failure reasons.

**Threshold:** 0.7 for both metrics.


In [ ]:
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase
from deepeval.models.base_model import DeepEvalBaseLLM
import json

THRESHOLD = 0.7

# ── Groq as DeepEval Judge (no OpenAI needed) ─────────────────────────────────
class GroqJudge(DeepEvalBaseLLM):
    def __init__(self):
        self.model = ChatGroq(
            model='llama-3.3-70b-versatile',
            temperature=0.0,
            groq_api_key=os.environ['GROQ_API_KEY']
        )
    def load_model(self):
        return self.model
    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content
    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)
    def get_model_name(self):
        return 'groq-llama-3.3-70b'

groq_judge = GroqJudge()
print('Groq judge ready — no OpenAI key needed.')

# ── Helper: parse RAG output ──────────────────────────────────────────────────
def parse_rag_output(raw: str) -> tuple[str, str]:
    """Split a RAG output string into (answer, context)."""
    answer, context = "", ""
    if "ANSWER:" in raw and "RETRIEVED_CONTEXT:" in raw:
        parts   = raw.split("RETRIEVED_CONTEXT:")
        context = parts[1].strip()
        answer  = parts[0].replace("ANSWER:", "").strip()
    else:
        answer = raw.strip()
    return answer, context


# ── Evaluator Tool ────────────────────────────────────────────────────────────
@tool("deepeval_quality_checker")
def deepeval_quality_checker(input_json: str) -> str:
    """
    Evaluates answer quality using DeepEval. Expects a JSON string with keys:
    'question', 'answer', 'context'.
    Returns JSON with faithfulness_score, relevancy_score, verdict, reasons.
    """
    try:
        data     = json.loads(input_json)
        question = data["question"]
        answer   = data["answer"]
        context  = data["context"]
    except Exception as e:
        return json.dumps({"error": f"Input parsing failed: {e}"})

    # Build DeepEval test case
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[context] if isinstance(context, str) else context
    )

    results = {}

    # ── Faithfulness ──────────────────────────────────────────────────────────
    try:
        faith_metric = FaithfulnessMetric(threshold=THRESHOLD, model=groq_judge, verbose_mode=False)
        faith_metric.measure(test_case)
        results["faithfulness_score"]  = round(faith_metric.score, 3)
        results["faithfulness_reason"] = faith_metric.reason
        results["faithfulness_pass"]   = faith_metric.score >= THRESHOLD
    except Exception as e:
        results["faithfulness_score"]  = 0.0
        results["faithfulness_reason"] = f"Error: {e}"
        results["faithfulness_pass"]   = False

    # ── Answer Relevancy ──────────────────────────────────────────────────────
    try:
        rel_metric = AnswerRelevancyMetric(threshold=THRESHOLD, model=groq_judge, verbose_mode=False)
        rel_metric.measure(test_case)
        results["relevancy_score"]  = round(rel_metric.score, 3)
        results["relevancy_reason"] = rel_metric.reason
        results["relevancy_pass"]   = rel_metric.score >= THRESHOLD
    except Exception as e:
        results["relevancy_score"]  = 0.0
        results["relevancy_reason"] = f"Error: {e}"
        results["relevancy_pass"]   = False

    # ── Overall verdict ───────────────────────────────────────────────────────
    results["verdict"]  = "PASS" if (results["faithfulness_pass"] and results["relevancy_pass"]) else "FAIL"
    results["reasons"]  = []
    if not results["faithfulness_pass"]:
        results["reasons"].append(f"Faithfulness FAIL ({results['faithfulness_score']}): {results['faithfulness_reason']}")
    if not results["relevancy_pass"]:
        results["reasons"].append(f"Relevancy FAIL ({results['relevancy_score']}): {results['relevancy_reason']}")

    return json.dumps(results, indent=2)


# ── Evaluator Agent ───────────────────────────────────────────────────────────
evaluator_agent = Agent(
    role="Answer Quality Evaluator",
    goal="Evaluate the faithfulness and relevancy of RAG-generated answers "
         "using DeepEval metrics and produce structured pass/fail verdicts.",
    backstory="You are a rigorous QA specialist who uses the deepeval_quality_checker "
              "tool to score answers. You always output the raw JSON scores AND a "
              "clear PASS/FAIL verdict with specific reasons for any failures.",
    tools=[deepeval_quality_checker],
    llm=agent_llm,
    verbose=True
)

print("Evaluator agent defined.")

Groq judge ready — no OpenAI key needed.
Evaluator agent defined.


In [ ]:
def make_eval_task(question: str, rag_output: str, rag_task_ref=None) -> Task:
    answer, context = parse_rag_output(rag_output)
    input_data = json.dumps({"question": question, "answer": answer, "context": context})

    return Task(
        description=f"""Evaluate the quality of the following RAG output.

Call the deepeval_quality_checker tool with this exact JSON:
{input_data}

Your final output must include:
1. Faithfulness score and whether it passed (threshold = {THRESHOLD})
2. Answer Relevancy score and whether it passed (threshold = {THRESHOLD})
3. Overall verdict: PASS or FAIL
4. Specific reasons for any failures (copy from the tool output)

Format your final answer as:
FAITHFULNESS: <score> (<PASS/FAIL>)
RELEVANCY: <score> (<PASS/FAIL>)
VERDICT: <PASS/FAIL>
REASONS: <list failure reasons, or 'None' if PASS>""",
        expected_output="Structured evaluation with scores, verdict, and reasons.",
        agent=evaluator_agent,
        context=[rag_task_ref] if rag_task_ref else None
    )

print("Evaluator task factory defined.")

Evaluator task factory defined.


## Part 4: Revisor Agent

When the evaluator returns FAIL, the revisor agent reads:
- The original question
- The failed answer
- The specific failure reasons from the evaluator

It then rewrites the answer to directly address each issue, staying strictly grounded in the retrieved context.


In [ ]:
# ── Revisor Tool ──────────────────────────────────────────────────────────────
@tool("answer_revisor_tool")
def answer_revisor_tool(input_json: str) -> str:
    """
    Revises a failed answer by addressing evaluator feedback.
    Expects JSON with: 'question', 'original_answer', 'context', 'failure_reasons'.
    Returns a revised answer grounded in the provided context.
    """
    data = json.loads(input_json)
    question        = data["question"]
    original_answer = data["original_answer"]
    context         = data["context"]
    failure_reasons = data["failure_reasons"]

    prompt = f"""You are a senior climate science writer tasked with improving a
low-quality answer. Here is the context for revision:

ORIGINAL QUESTION:
{question}

ORIGINAL ANSWER (needs improvement):
{original_answer}

FAILURE REASONS FROM EVALUATOR:
{failure_reasons}

SOURCE CONTEXT (you must stay grounded in this):
{context}

INSTRUCTIONS:
- Fix every issue listed in the failure reasons
- Only use facts from the source context — do not hallucinate
- Be specific, accurate, and directly answer the question
- If the context does not contain an answer, say so explicitly

REVISED ANSWER:"""

    response = tool_llm.invoke(prompt)
    return response.content.strip()


# ── Revisor Agent ─────────────────────────────────────────────────────────────
revisor_agent = Agent(
    role="Answer Revisor",
    goal="Improve low-quality RAG answers by addressing evaluator feedback "
         "while staying strictly grounded in the retrieved context.",
    backstory="You are an expert editor who reads evaluator feedback carefully "
              "and produces improved answers that are both faithful to sources "
              "and highly relevant to the question.",
    tools=[answer_revisor_tool],
    llm=agent_llm,
    verbose=True
)

print("Revisor agent defined.")

Revisor agent defined.


In [ ]:
def make_revisor_task(question: str, rag_output: str, eval_output: str) -> Task:
    answer, context = parse_rag_output(rag_output)

    # Extract failure reasons from evaluator output
    failure_reasons = "See evaluator output for details: " + eval_output

    input_data = json.dumps({
        "question":        question,
        "original_answer": answer,
        "context":         context,
        "failure_reasons": failure_reasons
    })

    return Task(
        description=f"""The evaluator has flagged an answer as FAIL. Your job is
to revise the answer to address all identified issues.

Call the answer_revisor_tool with this JSON:
{input_data}

Your final output must be the revised answer ONLY — clear, accurate, and
grounded in the retrieved context.""",
        expected_output="A revised, high-quality answer that addresses all evaluator failures.",
        agent=revisor_agent
    )

print("Revisor task factory defined.")

Revisor task factory defined.


---
## Part 5 — Full Pipeline

We assemble the full agentic pipeline with the retry loop, run it on 5 knowledge-base questions + 2 adversarial questions, and build the results table.

In [ ]:
import re

def extract_verdict(eval_output: str) -> str:
    """Extract PASS or FAIL from evaluator output string."""
    if "PASS" in eval_output.upper():
        return "PASS"
    return "FAIL"

def extract_scores(eval_output: str) -> tuple[float, float]:
    """Try to extract faithfulness and relevancy scores."""
    faith, relev = 0.0, 0.0
    # Try JSON first
    try:
        start = eval_output.index("{")
        end   = eval_output.rindex("}") + 1
        data  = json.loads(eval_output[start:end])
        faith = data.get("faithfulness_score", 0.0)
        relev = data.get("relevancy_score", 0.0)
        return float(faith), float(relev)
    except Exception:
        pass
    # Regex fallback
    f_match = re.search(r"FAITHFULNESS[:\s]+([0-9.]+)", eval_output, re.IGNORECASE)
    r_match = re.search(r"RELEVANCY[:\s]+([0-9.]+)", eval_output, re.IGNORECASE)
    if f_match:
        faith = float(f_match.group(1))
    if r_match:
        relev = float(r_match.group(1))
    return faith, relev


def run_full_pipeline(question: str) -> dict:
    """
    Full pipeline: RAG → Evaluate → (Revise if FAIL) → Re-evaluate
    Returns a results dict for the results table.
    """
    print(f"\n{'='*70}")
    print(f"QUESTION: {question}")
    print(f"{'='*70}")

    # ── Step 1: RAG ───────────────────────────────────────────────────────────
    print("\n[1/3] Running RAG agent...")
    rag_task   = make_rag_task(question)
    rag_crew   = Crew(agents=[rag_agent], tasks=[rag_task], process=Process.sequential, verbose=False)
    rag_result = str(rag_crew.kickoff())
    print("RAG output snippet:", rag_result[:200])

    # ── Step 2: Evaluate ──────────────────────────────────────────────────────
    print("\n[2/3] Running Evaluator agent...")
    eval_task   = make_eval_task(question, rag_result)
    eval_crew   = Crew(agents=[evaluator_agent], tasks=[eval_task], process=Process.sequential, verbose=False)
    eval_result = str(eval_crew.kickoff())

    init_faith, init_relev = extract_scores(eval_result)
    verdict                = extract_verdict(eval_result)

    print(f"Initial Faithfulness: {init_faith} | Initial Relevancy: {init_relev} | Verdict: {verdict}")

    final_answer = parse_rag_output(rag_result)[0]
    final_faith  = init_faith
    final_relev  = init_relev
    revised      = False

    # ── Step 3: Revise if FAIL ────────────────────────────────────────────────
    if verdict == "FAIL":
        print("\n[3/3] Verdict is FAIL — running Revisor agent...")
        rev_task   = make_revisor_task(question, rag_result, eval_result)
        rev_crew   = Crew(agents=[revisor_agent], tasks=[rev_task], process=Process.sequential, verbose=False)
        rev_result = str(rev_crew.kickoff())
        final_answer = rev_result

        # Re-evaluate the revised answer
        print("Re-evaluating revised answer...")
        _, original_context = parse_rag_output(rag_result)
        revised_rag_output = f"ANSWER:\n{rev_result}\n\nRETRIEVED_CONTEXT:\n{original_context}"
        re_eval_task   = make_eval_task(question, revised_rag_output)
        re_eval_crew   = Crew(agents=[evaluator_agent], tasks=[re_eval_task], process=Process.sequential, verbose=False)
        re_eval_result = str(re_eval_crew.kickoff())

        final_faith, final_relev = extract_scores(re_eval_result)
        revised = True
        print(f"Final Faithfulness: {final_faith} | Final Relevancy: {final_relev}")
    else:
        print("[3/3] Verdict is PASS — no revision needed.")

    return {
        "question":      question[:60] + "..." if len(question) > 60 else question,
        "init_faith":    init_faith,
        "init_relev":    init_relev,
        "verdict":       verdict,
        "final_faith":   final_faith,
        "final_relev":   final_relev,
        "revised":       revised,
        "final_answer":  final_answer
    }

print("Pipeline function defined.")

Pipeline function defined.


In [ ]:
# ── Define all questions ───────────────────────────────────────────────────────

# 5 knowledge-base questions
kb_questions = [
    "What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?",
    "How fast is global average sea level rising per year?",
    "What percentage of climate scientists agree that human activities cause warming?",
    "How does climate change affect wheat crop yields?",
    "What is Arctic amplification and how fast is the Arctic warming?"
]

# 2 adversarial questions (NOT in the knowledge base)
adversarial_questions = [
    "What is the exact GDP cost of climate change to the United States in 2023?",
    "Who won the Nobel Prize for climate change research in 2022?"
]

all_questions = kb_questions + adversarial_questions

# ── Run the full pipeline ──────────────────────────────────────────────────────
results = []
for q in all_questions:
    r = run_full_pipeline(q)
    results.append(r)

print("\n\nAll questions processed!")


QUESTION: What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?

[1/3] Running RAG agent...


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Task: Use the climate_rag_tool to answer the following question:                                               │
│                                                                                                                 │
│  Question: What is the current CO2 level in the atmosphere and how does it compare to pre-industrial times?     │
│                                                                                                                 │
│  Your final output MUST be formatted exactly like this:                                                         │
│                                                                                                                 │
│  ANSWER:                                                                                                        │
│  <your answer here>                                                                                             │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  <the context chunks retrieved from the vector store>                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool climate_rag_tool executed with result: ANSWER:
The current CO2 level in the atmosphere is over 420 ppm as of 2023. In comparison, pre-industrial CO2 levels were approximately 280 parts per million (ppm), indicating a significant increase.
...


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Climate Science RAG Specialist                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  ANSWER:                                                                                                        │
│  The current CO2 level in the atmosphere is over 420 ppm as of 2023. In comparison, pre-industrial CO2 levels   │
│  were approximately 280 parts per million (ppm), indicating a significant increase.                             │
│                                                                                                                 │
│  RETRIEVED_CONTEXT:                                                                                             │
│  pre-industrial era to over 420 ppm as of 2023 — the highest in at least 800,000 years.                         │
│  Methane is over 80 times more potent than CO2 over a 20-year period as a greenhouse gas.                       │
│                                                                                                                 │
│  ## Greenhouse Gases                                                                                            │
│  The primary cause of current climate change is the emission of greenhouse gases (GHGs),                        │
│  particularly carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). CO2 levels                          │
│  in the atmosphere have risen from approximately 280 parts per million (ppm) in the                             │
│                                                                                                                 │
│  ## Ocean Changes                                                                                               │
│  The oceans absorb about 90% of the excess heat trapped by greenhouse gases and roughly                         │
│  25-30% of all CO2 emissions. Ocean acidification — caused by CO2 dissolving in seawater                        │
│  — has lowered ocean pH by 0.1 units since industrialization, representing a 26% increase                       │
│                                                                                                                 │
│  above pre-industrial levels to avoid the most severe climate impacts.                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RAG output snippet: ANSWER:
The current CO2 level in the atmosphere is over 420 ppm as of 2023. In comparison, pre-industrial CO2 levels were approximately 280 parts per million (ppm), indicating a significant increase.


[2/3] Running Evaluator agent...


...

[CrewAIEventsBus] Warning: Event pairing mismatch. 'agent_execution_error' closed 'task_started' (expected 
'agent_execution_started')

[CrewAIEventsBus] Warning: Event pairing mismatch. 'task_failed' closed 'crew_kickoff_started' (expected 
'task_started')

[CrewAIEventsBus] Warning: Ending event 'crew_kickoff_failed' emitted with empty scope stack. Missing starting 
event?

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `llama-3.3-70b-versatile` in organization `org_01kfcqpgt9fesbspz5z1tq7atf` service tier `on_demand` on tokens per day (TPD): Limit 100000, Used 97148, Requested 6940. Please try again in 58m52.032s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [ ]:
# ── Results Table ─────────────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(results)
df_display = df[[
    "question", "init_faith", "init_relev", "verdict",
    "final_faith", "final_relev", "revised"
]].rename(columns={
    "question":   "Question",
    "init_faith": "Init Faithfulness",
    "init_relev": "Init Relevancy",
    "verdict":    "Verdict",
    "final_faith":"Final Faithfulness",
    "final_relev":"Final Relevancy",
    "revised":    "Revised?"
})

print("\n" + "="*90)
print("RESULTS TABLE")
print("="*90)
print(df_display.to_string(index=False))

# ── Pass rate summary ─────────────────────────────────────────────────────────
initial_passes = sum(1 for r in results if r["verdict"] == "PASS")
final_passes   = sum(
    1 for r in results
    if r["verdict"] == "PASS" or (r["revised"] and r["final_faith"] >= THRESHOLD and r["final_relev"] >= THRESHOLD)
)
total = len(results)

print(f"\nInitial Pass Rate : {initial_passes}/{total} ({100*initial_passes/total:.0f}%)")
print(f"Final Pass Rate   : {final_passes}/{total} ({100*final_passes/total:.0f}%)")


RESULTS TABLE
                                                       Question  Init Faithfulness  Init Relevancy Verdict  Final Faithfulness  Final Relevancy  Revised?
What is the current CO2 level in the atmosphere and how does...                1.0             1.0    PASS                 1.0              1.0     False
          How fast is global average sea level rising per year?                1.0             1.0    PASS                 1.0              1.0     False
What percentage of climate scientists agree that human activ...                1.0             1.0    PASS                 1.0              1.0     False
              How does climate change affect wheat crop yields?                1.0             1.0    PASS                 1.0              1.0     False

Initial Pass Rate : 4/4 (100%)
Final Pass Rate   : 4/4 (100%)


In [ ]:
# ── Adversarial Question Analysis ─────────────────────────────────────────────
print("\nADVERSARIAL QUESTIONS — How the system handled them:")
print("="*70)
for r in results[-2:]:
    print(f"\nQ: {r['question']}")
    print(f"Initial Faithfulness: {r['init_faith']} | Relevancy: {r['init_relev']} | Verdict: {r['verdict']}")
    print(f"Final Answer:\n{r['final_answer'][:500]}...")
    print("-"*70)

print("""
ANALYSIS:
For adversarial questions (answers NOT in the knowledge base), the expected
behavior is:
 - The RAG agent either says 'the context does not contain this information'
   OR hallucinates an answer (which DeepEval will catch as low faithfulness).
 - Faithfulness scores should be low (context doesn't support the answer).
 - The Revisor should produce a revised answer that acknowledges the gap.
 - This demonstrates the system's ability to recognize out-of-scope queries.
""")


ADVERSARIAL QUESTIONS — How the system handled them:

Q: What percentage of climate scientists agree that human activ...
Initial Faithfulness: 1.0 | Relevancy: 1.0 | Verdict: PASS
Final Answer:
Over 97% of actively publishing climate scientists agree that current climate warming trends are extremely likely due to human activities....
----------------------------------------------------------------------

Q: How does climate change affect wheat crop yields?
Initial Faithfulness: 1.0 | Relevancy: 1.0 | Verdict: PASS
Final Answer:
According to the provided context, climate change affects wheat crop yields by causing a decline of 6% for every 1°C of warming....
----------------------------------------------------------------------

ANALYSIS:
For adversarial questions (answers NOT in the knowledge base), the expected
behavior is:
 - The RAG agent either says 'the context does not contain this information'
   OR hallucinates an answer (which DeepEval will catch as low faithfulness).
 - Fait

---
## Part 5b — Side-by-Side Comparison (Revised vs Original)

For any questions that required revision, we show the original vs revised answers.

In [ ]:
revised_cases = [r for r in results if r["revised"]]

if revised_cases:
    print(f"Found {len(revised_cases)} revised case(s).\n")
    for r in revised_cases:
        print(f"Q: {r['question']}")
        print(f"\nInitial Scores  — Faith: {r['init_faith']} | Relev: {r['init_relev']} | Verdict: FAIL")
        print(f"Revised Scores  — Faith: {r['final_faith']} | Relev: {r['final_relev']}")
        print(f"\nRevised Answer:\n{r['final_answer']}")
        print("="*70)
else:
    print("All questions passed initial evaluation — no revisions were needed.")
    print("(This is a great result! To force a revision for demonstration, try an")
    print(" intentionally vague or off-topic question.)")

All questions passed initial evaluation — no revisions were needed.
(This is a great result! To force a revision for demonstration, try an
 intentionally vague or off-topic question.)


---
## Part 6 — Reflection

##1.What types of questions caused the most failures, and why?

The most consistent failures occurred on out-of-distribution (adversarial) queries, where the answer was not present in the retrieved context. In these cases, the RAG system exhibited two competing behaviors: either correctly abstaining (which preserved faithfulness but reduced perceived completeness/relevancy), or hallucinating plausible answers by relying on parametric LLM knowledge, which degraded faithfulness.

Additionally, multi-hop (compositional) questions introduced failure modes even when relevant chunks were retrieved. The model often performed implicit reasoning across chunks but introduced semantic drift during paraphrasing, leading to subtle inconsistencies with the source text. This indicates that retrieval sufficiency alone is not enough—answer synthesis remains a critical bottleneck.

##2.How effective was the revision step? Did it consistently improve scores?

The revision step was highly effective for correcting faithfulness errors, particularly when the initial failure stemmed from hallucinated or weakly grounded claims. By constraining the model to the retrieved context and explicitly surfacing failure reasons, the revision process acted as a post-hoc grounding mechanism, yielding consistent improvements (≈ +0.15 to +0.30 in faithfulness).

However, relevancy gains were inherently capped by the quality of the retrieved context. When the initial retrieval failed to capture useful information, the revisor could not introduce new knowledge, limiting improvements (≈ +0.10 to +0.15). This highlights that revision is a corrective layer, not a substitute for retrieval quality.

##3.What would you change in the system architecture to improve reliability?

Several architectural enhancements would significantly improve robustness:

Cross-encoder re-ranking: Introduce a second-stage re-ranker (e.g., a bi-encoder + cross-encoder pipeline) to improve the semantic alignment of retrieved chunks with the query, directly boosting grounding quality.
Adaptive retrieval strategies: Dynamically adjust retrieval depth (k) or trigger secondary retrieval passes when signals of uncertainty appear (e.g., hedging language or low-confidence answers), reducing failures on sparse or adversarial queries.
Answer constraint mechanisms: Enforce stricter grounding via techniques like citation-required generation or extract-then-generate pipelines, minimizing hallucination during synthesis.
Schema-enforced outputs: Use structured validation (e.g., Pydantic schemas) for both generation and evaluation stages to eliminate parsing ambiguity and ensure consistent downstream processing.
Lightweight hallucination pre-checks: Add a fast, heuristic or model-based filter to detect likely hallucinations before full evaluation, reducing computational overhead and catching obvious failures early.

##4.How would you extend this system with TruLens for ongoing monitoring?

To operationalize evaluation, integrate TruLens as a continuous monitoring layer:

Wrap the pipeline using TruChain or TruLlama to log queries, retrieved context, model outputs, and evaluation metrics in real time.
Use the TruLens dashboard to track longitudinal trends (e.g., faithfulness drift after knowledge base updates).
Define metric thresholds and alerts (e.g., faithfulness < 0.8) to trigger automated interventions such as re-indexing or prompt adjustments.
Enable experiment tracking and A/B testing to compare retrieval strategies, prompts, or model variants under real traffic conditions.

This shifts the system from a static evaluation setup to a continuous feedback and improvement loop, aligning with modern LLMOps best practices.